# 07 - Hyperparameter Tuning con Optuna

GridSearchCV exploró 27 combinaciones fijas de hiperparámetros
sin mejorar el modelo base. Optuna implementa búsqueda bayesiana:
cada trial informa al siguiente mediante un modelo probabilístico
del espacio de búsqueda, concentrando los intentos en las regiones
más prometedoras en lugar de exploración exhaustiva.

## 1. Setup

In [1]:
import optuna
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
import joblib
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

X_train = pd.read_csv('../data/X_train.csv')
X_test = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test = pd.read_csv('../data/y_test.csv').squeeze()

print("Datos cargados")
print(f"X_train: {X_train.shape}")

Datos cargados
X_train: (18101, 4)


## 2. Espacio de búsqueda

Se amplía el espacio respecto a GridSearchCV incorporando dos
hiperparámetros adicionales: subsample y colsample_bytree.

`n_estimators` (100-500): cantidad de árboles del ensemble.
Rango más amplio que GridSearch para explorar modelos más complejos.

`max_depth` (3-9): profundidad máxima de cada árbol. Valores altos
aumentan capacidad expresiva pero incrementan riesgo de overfitting.

`learning_rate` (0.01-0.3): tasa de aprendizaje. Controla cuánto
corrige cada árbol los errores del anterior.

`subsample` (0.6-1.0): fracción del dataset usada por cada árbol.
Introduce variabilidad entre árboles, reduciendo overfitting.
No estaba en GridSearchCV.

`colsample_bytree` (0.6-1.0): fracción de features usada por cada árbol.
Mecanismo análogo a subsample pero sobre columnas en lugar de filas.
No estaba en GridSearchCV.

`scale_pos_weight`: fijo, calculado como ratio no_peligrosos/peligrosos.
Compensa el desbalance de clases, no se incluye en la búsqueda.

In [2]:
scale_pos_weight = y_train.value_counts()[False] / y_train.value_counts()[True]

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'scale_pos_weight': scale_pos_weight,
        'random_state': 42,
        'eval_metric': 'logloss'
    }
    
    modelo = XGBClassifier(**params)
    scores = cross_val_score(modelo, X_train, y_train, 
                            cv=5, scoring='f1_macro', n_jobs=1)
    return scores.mean()

print("Función objetivo definida")

Función objetivo definida


## 3. Ejecución del estudio

50 trials con validación cruzada de 5 folds por trial.
Optuna selecciona los parámetros de cada trial basándose en los
resultados anteriores mediante el algoritmo TPE (Tree-structured
Parzen Estimator).

In [3]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\nMejor F1 Macro: {study.best_value:.4f}")
print(f"\nMejores parámetros:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

  0%|          | 0/50 [00:00<?, ?it/s]


Mejor F1 Macro: 0.7406

Mejores parámetros:
  n_estimators: 376
  max_depth: 4
  learning_rate: 0.22691701520478566
  subsample: 0.7607142383911873
  colsample_bytree: 0.7599685007024329


## 4. Evaluación del modelo optimizado

In [6]:
from sklearn.metrics import classification_report

xgb_optuna = XGBClassifier(
    n_estimators=study.best_params['n_estimators'],
    max_depth=study.best_params['max_depth'],
    learning_rate=study.best_params['learning_rate'],
    subsample=study.best_params['subsample'],
    colsample_bytree=study.best_params['colsample_bytree'],
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)

xgb_optuna.fit(X_train, y_train)
y_pred_optuna = xgb_optuna.predict(X_test)

print("XGBoost + Optuna")
print(classification_report(y_test, y_pred_optuna,
                            target_names=['No peligroso', 'Peligroso']))

XGBoost + Optuna
              precision    recall  f1-score   support

No peligroso       0.99      0.93      0.96      4274
   Peligroso       0.41      0.77      0.54       252

    accuracy                           0.93      4526
   macro avg       0.70      0.85      0.75      4526
weighted avg       0.95      0.93      0.94      4526



## 5. Comparativa final

Optuna logró una mejora moderada sobre el modelo base incorporando
subsample y colsample_bytree al espacio de búsqueda. Estos parámetros
introducen aleatoriedad controlada en cada árbol, reduciendo overfitting
y mejorando la generalización.

La mejora es real pero acotada, con 4 features básicas el modelo
llegó a su techo de performance. Features orbitales adicionales
(excentricidad, inclinación, semieje mayor) serían necesarias para
mejoras sustanciales.

El modelo final es XGBoost con parámetros de Optuna y umbral de
decisión 0.20 (definido en 06_shap.ipynb):

| Métrica | Valor |
|---------|-------|
| F1 Macro | 0.75 |

In [7]:
import joblib

joblib.dump(
    {'modelo': xgb_optuna, 'umbral': 0.20},
    '../data/xgboost_final.pkl'
)
print("Modelo final guardado: XGBoost + Optuna, umbral 0.20")

Modelo final guardado: XGBoost + Optuna, umbral 0.20
